### Dataset construction
1. `deep_learning/datasets/build_splits.py`
2. `deep_learning/datasets/build_context_to_idiom.py`
3. `deep_learning/datasets/build_idiom_to_meaning.py`
4. `deep_learning/datasets/build_arabic_context_to_idiom.py`
5. `deep_learning/datasets/build_surface_to_canonical.py`
6. `deep_learning/datasets/build_dataset_inventory.py`
7. `deep_learning/datasets/build_idiom_detection.py`
8. `deep_learning/datasets/build_idiom_bank.py`


# IdiomX Dataset Splitting

This notebook creates leakage-safe dataset splits for the IdiomX project.

## Goals
- Load the official IdiomX dataset
- Split data at the idiom level
- Prevent train/validation/test leakage
- Save reusable base splits for downstream tasks

In [3]:
# 1. Setup
# =========================================

from pathlib import Path
import warnings
import pandas as pd

warnings.filterwarnings("ignore")

BASE_DIR = Path("..").resolve()
DATA_RAW_DIR = BASE_DIR / "data" / "raw"
DATA_SPLITS_DIR = BASE_DIR / "data" / "splits"

DATA_RAW_DIR.mkdir(parents=True, exist_ok=True)
DATA_SPLITS_DIR.mkdir(parents=True, exist_ok=True)

LOCAL_DATA_PATH = DATA_RAW_DIR / "idiomx_official.csv"

print("BASE_DIR       :", BASE_DIR)
print("DATA_RAW_DIR   :", DATA_RAW_DIR)
print("DATA_SPLITS_DIR:", DATA_SPLITS_DIR)
print("LOCAL_DATA_PATH:", LOCAL_DATA_PATH)

BASE_DIR       : C:\Users\ayman\Documents\IdiomX\github_idiomX\IdiomX
DATA_RAW_DIR   : C:\Users\ayman\Documents\IdiomX\github_idiomX\IdiomX\data\raw
DATA_SPLITS_DIR: C:\Users\ayman\Documents\IdiomX\github_idiomX\IdiomX\data\splits
LOCAL_DATA_PATH: C:\Users\ayman\Documents\IdiomX\github_idiomX\IdiomX\data\raw\idiomx_official.csv


In [4]:
# 2. Download official dataset if missing
# =========================================

if not LOCAL_DATA_PATH.exists():
    print("Local dataset not found. Downloading from Hugging Face...")

    from datasets import load_dataset

    ds = load_dataset("aymansharara/IdiomX")

    if "train" in ds:
        df = ds["train"].to_pandas()
    else:
        parts = [ds[split_name].to_pandas() for split_name in ds.keys()]
        df = pd.concat(parts, ignore_index=True)

    df.to_csv(LOCAL_DATA_PATH, index=False, encoding="utf-8-sig")
    print("Saved locally:", LOCAL_DATA_PATH)
    print("Shape:", df.shape)

else:
    print("Local dataset already exists. Skipping download.")

Local dataset already exists. Skipping download.


In [5]:
# 3. Load dataset
# =========================================

df = pd.read_csv(LOCAL_DATA_PATH, low_memory=False)

print("Dataset loaded successfully.")
print("Shape:", df.shape)

Dataset loaded successfully.
Shape: (123336, 34)


In [6]:
# 4. Quick split-key check
# =========================================

for col in ["idiom", "idiom_canonical", "idiom_surface"]:
    print(f"\nColumn: {col}")
    print("Unique:", df[col].nunique(dropna=True))
    print("Nulls :", df[col].isnull().sum())


Column: idiom
Unique: 15411
Nulls : 0

Column: idiom_canonical
Unique: 14986
Nulls : 184

Column: idiom_surface
Unique: 27751
Nulls : 1


## 5. Fix Split Key (idiom_canonical)

We use `idiom_canonical` as the official split key.

Missing values are replaced with `idiom` to ensure:
- no data loss
- consistent grouping
- leakage-safe splitting

In [7]:
# 5. Fix canonical key
# =========================================

df["idiom_canonical"] = df["idiom_canonical"].fillna(df["idiom"])

print("Nulls after fix:", df["idiom_canonical"].isnull().sum())

Nulls after fix: 0


## 6. Canonical Idiom Distribution

We analyze how many samples exist per canonical idiom to understand:
- data imbalance
- rare idioms
- split feasibility

In [8]:
# 6. Frequency analysis
# =========================================

idiom_counts = df["idiom_canonical"].value_counts()

print("Total unique idioms:", idiom_counts.shape[0])
print("\nTop 10 most frequent idioms:")
print(idiom_counts.head(10))

print("\nStatistics:")
print(idiom_counts.describe())

Total unique idioms: 15009

Top 10 most frequent idioms:
idiom_canonical
dance with the one that brought you     88
Bob's your uncle                        48
worship the porcelain god               40
you kiss your mother with that mouth    40
so help me God                          32
play the devil with                     32
time heals all wounds                   32
poverty is a state of mind              32
let it be                               32
may the Force be with you               32
Name: count, dtype: int64

Statistics:
count    15009.000000
mean         8.217470
std          1.632439
min          8.000000
25%          8.000000
50%          8.000000
75%          8.000000
max         88.000000
Name: count, dtype: float64


## 7. Create Base Split (idiom_canonical)

We split at idiom level to avoid leakage:
- 80% train
- 10% validation
- 10% test

In [9]:
# 7. Canonical split
# =========================================

from sklearn.model_selection import train_test_split

unique_idioms = df["idiom_canonical"].unique()

train_ids, temp_ids = train_test_split(
    unique_idioms,
    test_size=0.2,
    random_state=42
)

val_ids, test_ids = train_test_split(
    temp_ids,
    test_size=0.5,
    random_state=42
)

train_df = df[df["idiom_canonical"].isin(train_ids)].copy()
val_df   = df[df["idiom_canonical"].isin(val_ids)].copy()
test_df  = df[df["idiom_canonical"].isin(test_ids)].copy()

print("Train:", train_df.shape)
print("Val  :", val_df.shape)
print("Test :", test_df.shape)

Train: (98696, 34)
Val  : (12384, 34)
Test : (12256, 34)


In [10]:
# 8. Leakage check
# =========================================

train_set = set(train_df["idiom_canonical"])
val_set   = set(val_df["idiom_canonical"])
test_set  = set(test_df["idiom_canonical"])

print("Train ∩ Val :", len(train_set & val_set))
print("Train ∩ Test:", len(train_set & test_set))
print("Val ∩ Test  :", len(val_set & test_set))

Train ∩ Val : 0
Train ∩ Test: 0
Val ∩ Test  : 0


In [11]:
# 9. Save splits
# =========================================

train_df.to_csv(DATA_SPLITS_DIR / "base_train.csv", index=False)
val_df.to_csv(DATA_SPLITS_DIR / "base_val.csv", index=False)
test_df.to_csv(DATA_SPLITS_DIR / "base_test.csv", index=False)

print("Base splits saved.")

Base splits saved.


## Split Reproducibility via Python Pipeline

In addition to the notebook-based split procedure, the same split can be reproduced using the standalone Python script `build_splits.py`.

This ensures the split process is reusable from both:
- Jupyter notebooks
- command line execution

In [12]:
import importlib.util

module_path = BASE_DIR / "deep_learning" / "datasets" / "build_splits.py"

spec = importlib.util.spec_from_file_location("build_splits", module_path)
build_splits_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(build_splits_module)

split_out = build_splits_module.build_splits()

print("Saved to:", split_out["output_dir"])
print("Train shape:", split_out["train_df"].shape)
print("Val shape  :", split_out["val_df"].shape)
print("Test shape :", split_out["test_df"].shape)

Loading official IdiomX dataset...
Total usable rows : 109182
Unique idioms     : 15009

Split sizes:
Train rows      : 87355 | idioms: 12007
Validation rows : 10943 | idioms: 1501
Test rows       : 10884 | idioms: 1501

Usage-label balance:
train {'idiomatic': 44130, 'literal': 43225}
validation {'idiomatic': 5547, 'literal': 5396}
test {'idiomatic': 5504, 'literal': 5380}

Overlap checks:
Train ∩ Validation : 0
Train ∩ Test       : 0
Validation ∩ Test  : 0

Saved base splits to: C:\Users\ayman\Documents\IdiomX\github_idiomX\IdiomX\data\splits\base
Saved to: C:\Users\ayman\Documents\IdiomX\github_idiomX\IdiomX\data\splits\base
Train shape: (87355, 34)
Val shape  : (10943, 34)
Test shape : (10884, 34)


## Surface Normalization Split (Surface → Canonical)

Unlike other tasks, this task requires splitting based on surface forms (`idiom_surface`) rather than canonical idioms.

This ensures the model does not see the same surface form in multiple splits.

In [13]:
module_path = BASE_DIR / "deep_learning" / "datasets" / "build_surface_to_canonical.py"

spec = importlib.util.spec_from_file_location("surface_builder", module_path)
surface_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(surface_module)

surface_out = surface_module.build_surface_to_canonical_dataset()

print("Saved to:", surface_out["output_dir"])
print("Train:", surface_out["train_df"].shape)
print("Val  :", surface_out["validation_df"].shape)
print("Test :", surface_out["test_df"].shape)

Loading official IdiomX dataset for surface split...

Processing train...
train: 22,200 rows

Processing validation...
validation: 2,775 rows

Processing test...
test: 2,775 rows

Saved surface dataset to: C:\Users\ayman\Documents\IdiomX\github_idiomX\IdiomX\deep_learning\datasets\surface_normalization
Saved to: C:\Users\ayman\Documents\IdiomX\github_idiomX\IdiomX\deep_learning\datasets\surface_normalization
Train: (22200, 15)
Val  : (2775, 15)
Test : (2775, 15)


## Task 1 — Idiom Detection

This task is a binary classification problem:

- Input: sentence containing an idiom candidate  
- Output: whether the usage is idiomatic or literal  

The dataset is built from the base canonical split.

In [14]:
module_path = BASE_DIR / "deep_learning" / "datasets" / "build_idiom_detection.py"

spec = importlib.util.spec_from_file_location("idiom_detection_builder", module_path)
idiom_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(idiom_module)

idiom_out = idiom_module.build_idiom_detection_dataset()

print("Saved to:", idiom_out["output_dir"])
print("Train:", idiom_out["train_df"].shape)
print("Val  :", idiom_out["validation_df"].shape)
print("Test :", idiom_out["test_df"].shape)


Processing train...
train: 87,320 rows
Label distribution: {1: 44128, 0: 43192}

Processing validation...
validation: 10,943 rows
Label distribution: {1: 5551, 0: 5392}

Processing test...
test: 10,883 rows
Label distribution: {1: 5505, 0: 5378}

Saved idiom detection dataset to: C:\Users\ayman\Documents\IdiomX\github_idiomX\IdiomX\deep_learning\datasets\idiom_detection
Saved to: C:\Users\ayman\Documents\IdiomX\github_idiomX\IdiomX\deep_learning\datasets\idiom_detection
Train: (87320, 16)
Val  : (10943, 16)
Test : (10883, 16)


## Task 2 — Context to Idiom

This task predicts the correct idiom given a sentence context.

- Input: sentence (context)
- Output: idiom (canonical form)

The dataset supports different quality modes:
- full: use all data
- soft: moderate filtering
- strict: high-quality subset

In [15]:
module_path = BASE_DIR / "deep_learning" / "datasets" / "build_context_to_idiom.py"

spec = importlib.util.spec_from_file_location("context_builder", module_path)
context_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(context_module)

context_out = context_module.build_context_to_idiom_dataset(mode="full")

print("Saved to:", context_out["output_dir"])
print("Train:", context_out["train_df"].shape)
print("Val  :", context_out["validation_df"].shape)
print("Test :", context_out["test_df"].shape)


Processing train...
train: 87,355 rows

Processing validation...
validation: 10,943 rows

Processing test...
test: 10,884 rows

Saved context → idiom dataset to: C:\Users\ayman\Documents\IdiomX\github_idiomX\IdiomX\deep_learning\datasets\context_to_idiom
Saved to: C:\Users\ayman\Documents\IdiomX\github_idiomX\IdiomX\deep_learning\datasets\context_to_idiom
Train: (87355, 13)
Val  : (10943, 13)
Test : (10884, 13)


## Task 3 — Arabic Context to Idiom

This task predicts the correct English idiom from an Arabic sentence.

- Input: Arabic sentence
- Output: English idiom (canonical form)

This task enables cross-lingual understanding and translation.

In [16]:
module_path = BASE_DIR / "deep_learning" / "datasets" / "build_arabic_context_to_idiom.py"

spec = importlib.util.spec_from_file_location("arabic_builder", module_path)
arabic_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(arabic_module)

arabic_out = arabic_module.build_arabic_context_to_idiom_dataset()

print("Saved to:", arabic_out["output_dir"])
print("Train:", arabic_out["train_df"].shape)
print("Val  :", arabic_out["validation_df"].shape)
print("Test :", arabic_out["test_df"].shape)


Processing train...
train: 55,990 rows

Processing validation...
validation: 7,024 rows

Processing test...
test: 6,876 rows

Saved Arabic context → idiom dataset to: C:\Users\ayman\Documents\IdiomX\github_idiomX\IdiomX\deep_learning\datasets\arabic_context_to_idiom
Saved to: C:\Users\ayman\Documents\IdiomX\github_idiomX\IdiomX\deep_learning\datasets\arabic_context_to_idiom
Train: (55990, 13)
Val  : (7024, 13)
Test : (6876, 13)


## Task 5 — Idiom to Meaning

This task maps an idiom to its meaning.

- Input: Idiom (canonical form)
- Output: Meaning (English + Arabic)

This task serves as a semantic grounding benchmark and supports interpretation and explanation.

In [17]:
module_path = BASE_DIR / "deep_learning" / "datasets" / "build_idiom_to_meaning.py"

spec = importlib.util.spec_from_file_location("meaning_builder", module_path)
meaning_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(meaning_module)

meaning_out = meaning_module.build_idiom_to_meaning_dataset()

print("Saved to:", meaning_out["output_dir"])
print("Train:", meaning_out["train_df"].shape)
print("Val  :", meaning_out["validation_df"].shape)
print("Test :", meaning_out["test_df"].shape)


Processing train...
train: 11,183 rows

Processing validation...
validation: 1,396 rows

Processing test...
test: 1,392 rows

Saved idiom → meaning dataset to: C:\Users\ayman\Documents\IdiomX\github_idiomX\IdiomX\deep_learning\datasets\idiom_to_meaning
Saved to: C:\Users\ayman\Documents\IdiomX\github_idiomX\IdiomX\deep_learning\datasets\idiom_to_meaning
Train: (11183, 11)
Val  : (1396, 11)
Test : (1392, 11)


## Idiom Bank

The idiom bank is a compact canonical reference table containing one row per idiom.

It is useful for:
- dataset summarization
- lookup and inspection
- reporting vocabulary size and metadata coverage

In [18]:
module_path = BASE_DIR / "deep_learning" / "datasets" / "build_idiom_bank.py"

spec = importlib.util.spec_from_file_location("idiom_bank_builder", module_path)
idiom_bank_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(idiom_bank_module)

idiom_bank_out = idiom_bank_module.build_idiom_bank_dataset()

print("Saved to:", idiom_bank_out["output_dir"])
print("Shape:", idiom_bank_out["idiom_bank"].shape)

Loading official IdiomX dataset...
Total rows: 123,336
Grouping by canonical idioms...
Unique idioms: 15,009
Saved idiom bank to: C:\Users\ayman\Documents\IdiomX\github_idiomX\IdiomX\deep_learning\datasets\idiom_bank
Saved to: C:\Users\ayman\Documents\IdiomX\github_idiomX\IdiomX\deep_learning\datasets\idiom_bank
Shape: (15009, 13)


## Dataset Inventory

The dataset inventory summarizes all task-specific datasets after preprocessing.

It provides a compact overview of:
- split sizes
- number of columns
- unique inputs and targets
- class balance for classification tasks

In [19]:
module_path = BASE_DIR / "deep_learning" / "datasets" / "build_dataset_inventory.py"

spec = importlib.util.spec_from_file_location("inventory_builder", module_path)
inventory_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(inventory_module)

inventory_df = inventory_module.build_dataset_inventory()
display(inventory_df)

Building dataset inventory...

✔ idiom_detection_train
✔ idiom_detection_validation
✔ idiom_detection_test
✔ context_to_idiom_train
✔ context_to_idiom_validation
✔ context_to_idiom_test
✔ arabic_context_to_idiom_train
✔ arabic_context_to_idiom_validation
✔ arabic_context_to_idiom_test
✔ surface_normalization_train
✔ surface_normalization_validation
✔ surface_normalization_test
✔ idiom_to_meaning_train
✔ idiom_to_meaning_validation
✔ idiom_to_meaning_test

Saved inventory to: C:\Users\ayman\Documents\IdiomX\github_idiomX\IdiomX\deep_learning\datasets\dataset_inventory.csv


,dataset,rows,columns,unique_inputs,unique_targets,label_0,label_1
0,idiom_detection_train,87320,16,87316,NaN,43192.0,44128.0
1,idiom_detection_validation,10943,16,10943,NaN,5392.0,5551.0
2,idiom_detection_test,10883,16,10883,NaN,5378.0,5505.0
3,context_to_idiom_train,87355,13,87316,12007.0,NaN,NaN
4,context_to_idiom_validation,10943,13,10943,1501.0,NaN,NaN
5,context_to_idiom_test,10884,13,10883,1501.0,NaN,NaN
6,arabic_context_to_idiom_train,55990,13,55987,10936.0,NaN,NaN
7,arabic_context_to_idiom_validation,7024,13,7024,1364.0,NaN,NaN
8,arabic_context_to_idiom_test,6876,13,6876,1361.0,NaN,NaN
9,surface_normalization_train,22200,15,22200,12891.0,NaN,NaN
